# Miniproyecto 2 · Aprendizaje supervisado: de la recta a las estrellas
*Curso de Machine Learning · Miniproyecto · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

Vas a ajustar regresiones de complejidad creciente, ver el **sobreajuste** y cómo lo controla la **penalización**, y
terminar clasificando **estrellas variables** a partir de cómo cambia su brillo.

**Entrega:** tu copia en Drive, con todas las celdas ejecutadas y comentadas (100 puntos). No necesita GPU.

In [ ]:
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Elige los datos ──────────────────────────────────────────────────────────────────────────────
# "abiertos":  datos públicos o generados aquí (funciona siempre).
# "diplomado": los datos originales del miniproyecto del diplomado; sube los CSV a tu Drive, en
#              MyDrive/MiAtlas-ML/datos-diplomado/ (solo si tienes esos archivos).
DATOS = "abiertos"
if DATOS == "diplomado":
    if EN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        CARPETA_DATOS = Path("/content/drive/MyDrive/MiAtlas-ML/datos-diplomado")
    else:
        CARPETA_DATOS = Path("datos-diplomado")
print("Datos:", DATOS)

## Parte 1 · Regresión lineal por solución cerrada (25 puntos)

1. Con `np.random` y una **semilla**, genera 100 valores de `x` entre 0 y 3 e `y = 5 + 2x + u`, con `u` uniforme entre
   -2 y 5 (5).
2. Ajusta la recta con la **solución cerrada** usando `numpy.linalg` (por ejemplo, `lstsq`) y calcula el **error
   cuadrático medio** (10).
3. Grafica los datos y la recta ajustada (10).

In [ ]:
# ← tu código (Parte 1)

## Parte 2 · Regresión polinomial sobre una curva de luz (20 puntos)

Una **curva de luz** es el brillo de una estrella en el tiempo.

1. Carga la curva y toma una muestra de 200 puntos; grafícala (5).
2. Ajusta regresiones polinomiales de varios grados (`PolynomialFeatures` + `LinearRegression`) (5).
3. Grafica los modelos sobre los datos (5).
4. ¿Qué pasa al subir el grado? Usa las palabras **complejidad**, **ajuste** y **sobreajuste** (5).

In [ ]:
if DATOS == "diplomado":
    curva = pd.read_csv(CARPETA_DATOS / "mp2_curva_de_luz.csv")   # la primera columna sin nombre es el índice
else:
    # Una estrella pulsante simulada: período de 0,6 días, observada en momentos irregulares, con ruido
    rng = np.random.default_rng(0)
    tiempo = np.sort(rng.uniform(0, 1.8, 1000))
    magnitud = 0.4 * np.sin(2 * np.pi * tiempo / 0.6) + 0.15 * np.sin(4 * np.pi * tiempo / 0.6) + rng.normal(0, 0.08, 1000)
    curva = pd.DataFrame({"tiempo": tiempo, "magnitud": magnitud})
curva.plot.scatter("tiempo", "magnitud", s=4, title="Curva de luz");

In [ ]:
# ← tu código (Parte 2)

## Parte 3 · Ridge y Lasso (20 puntos)

Con la misma curva y un polinomio de grado **mayor que 5**:

1. Ajusta **Ridge** con varios valores de `alpha` (5) y 2. **Lasso** con varios `alpha` (5).
3. Grafica los modelos (5).
4. Muestra el intercepto y los coeficientes: ¿qué pasa al subir `alpha` en cada uno? ¿Por qué Lasso deja
   coeficientes en cero? (5)

In [ ]:
# ← tu código (Parte 3)

## Parte 4 · Clasificar estrellas variables (35 puntos)

1. Carga, explora, transforma (¿conviene el logaritmo del período?) y separa en entrenamiento y prueba (10).
2. Análisis descriptivo y preprocesamiento adecuado (escalar, clases desequilibradas…) (5).
3. Entrena **dos clasificadores** a elección; varía **dos hiperparámetros** de cada uno y comenta su efecto (5).
4. Entrena **KNN** y elige `n_neighbors` con experimentos (validación cruzada) (5).
5. Compara todos con **matrices de confusión** y precisión, *recall*, exactitud y F1 (10).

In [ ]:
# Estrellas variables
if DATOS == "diplomado":
    estrellas = pd.read_csv(CARPETA_DATOS / "mp2_estrellas.csv")        # características de curvas de luz
    estrellas = estrellas.rename(columns={"Class": "clase"})
else:
    # GCVS: Catálogo General de Estrellas Variables (Instituto Sternberg, CC BY 4.0), vía Hugging Face
    from datasets import load_dataset
    gcvs = load_dataset("juliensimon/gcvs-variable-stars")["train"].to_pandas()
    familias = {"EA": "eclipsante", "EB": "eclipsante", "EW": "eclipsante", "M": "período largo", "SRA": "período largo",
                "SRB": "período largo", "RRAB": "RR Lyrae", "RRC": "RR Lyrae", "DCEP": "cefeida", "DSCT": "delta Scuti",
                "HADS": "delta Scuti"}
    gcvs["clase"] = gcvs["variable_type"].str.split(":").str[0].map(familias)
    # En el GCVS, una magnitud mínima entre paréntesis («(») es directamente la amplitud
    gcvs["amplitud"] = np.where(gcvs["magnitude_min_flag"].eq("("), gcvs["magnitude_min"],
                                gcvs["magnitude_min"] - gcvs["magnitude_max"])
    estrellas = gcvs.dropna(subset=["clase", "period_days", "amplitud", "magnitude_max"])[
        ["clase", "period_days", "amplitud", "magnitude_max"]]
    estrellas = estrellas[estrellas["amplitud"] > 0]
    if PRUEBA_RAPIDA:
        estrellas = estrellas.sample(3000, random_state=0)
print(estrellas.shape)
print(estrellas["clase"].value_counts())

In [ ]:
# ← tu código (Parte 4)

*Tus conclusiones:*

**Fuentes:** [NumPy · `linalg.lstsq`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html) · [scikit-learn · regresión polinomial](https://scikit-learn.org/stable/modules/linear_model.html#polynomial-regression-extending-linear-models-with-basis-functions) · [scikit-learn · Ridge y Lasso](https://scikit-learn.org/stable/modules/linear_model.html) · [scikit-learn · KNN](https://scikit-learn.org/stable/modules/neighbors.html#classification) · [GCVS (Instituto Sternberg)](http://www.sai.msu.su/gcvs/gcvs/) · [gcvs-variable-stars en Hugging Face (CC BY 4.0)](https://huggingface.co/datasets/juliensimon/gcvs-variable-stars)